In [37]:
from itertools import combinations
from scipy.special import comb
import numpy as np
from lotteria import Lotteria

def find_row(array, matrix):
    for i, row in enumerate(matrix):
        if np.all(np.isin(array, row)):
            return i
    return -1

file_estrazione = "./data/estrazioni_2026_2006.csv"
ll = Lotteria(file_estrazione)
# Matrice di input
storico_estrazioni = ll.carica_storico(1000)
len_estrazioni = len(storico_estrazioni)

vettore_combinazioni = [comb(6, k, exact=True) for k in range(1,7)] # vettore coefficienti binomiali di 6 su k

size_intervallo = 60
max_basi = len_estrazioni - size_intervallo


# Matrice di output
matrice_ritardi = [[], [], [], [], [], []]

#for e, base in enumerate(basi):
print("Calcolo ritardi per ogni estrazione:")
for e in range(max_basi):
    print("Vettore di input:", e, end='\r')
    base = storico_estrazioni[e, :].copy()
    #print(base)
          
    matrice_intervallo = storico_estrazioni[e+1:e+1+size_intervallo]  # Prende le righe successive alla prima come matrice di intervallo
    # print("Matrice intervallo:")
    # print(matrice_intervallo)
        
    combinazioni_base = [list(combinations(base, k)) for k in range(1,7)] # combinazioni della base
    vettore_ritardi = np.full((len(base.ravel())), -1, dtype=np.int64)
    
    # print("Calcolo ritardi:")
    for i in range(1, len(base)+1):
        #print("Numero di combinazioni: ", vettore_combinazioni[i-1])
        vettore_ritardi = np.full(vettore_combinazioni[i-1], -1, dtype=np.int64)
        for j, combinazione in enumerate(combinazioni_base[i-1]):
            riga = find_row(combinazione, matrice_intervallo)
            # if riga > 0: 
            #     print(f"Combinazione con {riga} ritardi: ", *tuple(int(n) for n in combinazione))
            # else:
            #     print(f"Combinazione con 0 ritardi: ", *tuple(int(n) for n in combinazione))
            vettore_ritardi[j] = riga
        # print("Ritardi per combinazioni di lunghezza ", i, ":")
        # print(vettore_ritardi)
        matrice_ritardi[i-1].append(vettore_ritardi) # matrice_ritardi[i-1][0] = vettore_ritardi


# Stampa la matrice dei ritardi 

print("\nProduzione file ritardi:")
for i, ritardo in enumerate(matrice_ritardi):
    found = 0
    nome_file_out = f"Ritardi_{i+1}_{size_intervallo}.csv"   
    #print(nome_file_out)
    with open(nome_file_out, 'w') as f:
        for estrazione in ritardo:
            if not np.all(estrazione == -1): 
                found += 1
                r = ';'.join(str(element) for element in sorted(estrazione))  # Sostituisce -1 con 0 per la stampa
                f.write(r + '\n')
    # for estrazione in ritardo:
    #     if not np.all(estrazione == -1): 
    #         found += 1
    #         r = ';'.join(str(element) for element in estrazione)  # Sostituisce -1 con 0 per la stampa
    #         print(r)
    if found > 0: print("Trovate ", found, " combinazioni in classe ", i+1, " con ritardi contenuti nell'intervallo. Out su file: ", nome_file_out)
#print(matrice_ritardi)

Calcolo ritardi per ogni estrazione:
Vettore di input: 939
Produzione file ritardi:
Trovate  940  combinazioni in classe  1  con ritardi contenuti nell'intervallo. Out su file:  Ritardi_1_60.csv
Trovate  898  combinazioni in classe  2  con ritardi contenuti nell'intervallo. Out su file:  Ritardi_2_60.csv
Trovate  157  combinazioni in classe  3  con ritardi contenuti nell'intervallo. Out su file:  Ritardi_3_60.csv
Trovate  3  combinazioni in classe  4  con ritardi contenuti nell'intervallo. Out su file:  Ritardi_4_60.csv
